# 03 - Đánh giá cuối và phân tích lỗi

Chỉ dùng test sau khi đã chọn cấu hình bằng validation. Báo cáo Accuracy, Macro-F1, classification report và confusion matrix.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_dataloaders
from src.model import build_resnet18
from src.train import run_epoch
metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
labels = metadata.sort_values('label').drop_duplicates('label')['class_name'].tolist()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaders = make_dataloaders(metadata, augmentation=True)
model = build_resnet18(len(labels), freeze_backbone=False)
checkpoint = torch.load(PROJECT_ROOT / 'checkpoints' / 'E3_resnet_finetune_aug.pt', map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.to(device)
metrics = run_epoch(model, loaders['test'], torch.nn.CrossEntropyLoss(), device)
print(metrics)

In [ ]:
model.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for images, target in loaders['test']:
        prediction = model(images.to(device)).argmax(1).cpu()
        y_true.extend(target.tolist()); y_pred.extend(prediction.tolist())
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
matrix = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(14, 12)); sns.heatmap(matrix, cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.xlabel('Dự đoán'); plt.ylabel('Thật')
plt.tight_layout(); plt.savefig(PROJECT_ROOT / 'outputs' / 'confusion_matrix_test.png', dpi=160); plt.show()